# Step 4 — Reactivity Risk Φ_R (Calculations R1–R2)

**Purpose:** Compute an empirical proxy for the reactivity risk objective Φ_R for every memory sequence.

- **R1** — CDRH3 feature-based reactivity model: logistic regression predicting naive vs memory membership from CDRH3 biophysical features (hydrophobicity H_H3, net charge Q_H3, CDR3 length L_H3, aromatic fraction Y_H3) and V-gene covariates. High Φ_R = CDRH3 features associated with autoreactivity elimination (sequences with these features are depleted in memory relative to naive). Positive control: IGHV4-34 (encodes anti-I antigen autoreactivity via CDR1/CDR2).
- **R2** — Per-germline Φ_R shift naive → memory: for each IGHV germline gene, how much does mean Φ_R decrease from naive to memory? A large negative shift indicates that autoreactive sequences from that germline were preferentially cleared during tolerance checkpoints. IGHV4-34 is expected to show the largest depletion.

**Biological rationale:**
Autoreactive B cells are eliminated at several checkpoints:
1. Central tolerance (bone marrow): eliminates high-affinity self-reactors
2. Peripheral tolerance (naive pool, pre-GC): further anergy/deletion
3. GC selection: clones with cross-reactive (auto-)specificities may be eliminated

The aggregate effect: naive B cells with autoreactive CDRH3 features (high hydrophobicity, positive charge, long CDR3, high aromatic content) are progressively depleted from the memory compartment. Logistic regression captures this depletion signature as a feature weight vector, yielding an empirical Φ_R score.

**CDRH3 features (from Wardemann et al., Kohler et al.):**
- **H_H3**: mean Kyte-Doolittle hydrophobicity of CDR3 residues — high values correlate with polyreactivity
- **Q_H3**: net charge (#K+#R+#H − #D−#E) — positive values correlate with anti-DNA autoreactivity
- **L_H3**: CDR3 length (aa) — longer CDR3 enriched in autoreactive repertoires
- **Y_H3**: aromatic residue fraction (#F+#W+#Y / L) — correlates with polyreactivity

**Adaptations from Step 3 findings:**
- IgG sequences show lower Φ_A (more affinity-selected) than IgM → Φ_R is computed on all memory sequences but stratified by isotype class for R2 germline analysis
- IGHV4-39 and IGHV3-23 show strongest affinity selection signal → these should show moderate Φ_R shift (GC-selected, not autoreactivity-depleted)
- Isotype subclasses are collapsed to class level throughout

**Inputs:** `processed/aligned_master.parquet`  
**Outputs:** `results/tables/phi_r_scores.parquet`, `results/tables/phi_r_shift_by_germline.csv`, `results/figures/fig_r1_*.png`, `results/figures/fig_r2_*.png`

In [ ]:
import polars as pl
import numpy as np
import math
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample

In [ ]:
DATA_DIR  = Path("/home/jovyan/shared/Benjamin/LineageAtlas/pairplex_paper/")
PROC_DIR  = DATA_DIR / "processed"
RESULTS   = DATA_DIR / "results"
FIGURES   = RESULTS / "figures"
TABLES    = RESULTS / "tables"

MASTER_FILE = PROC_DIR / "aligned_master.parquet"
KEY = "seq_name"

print("Paths OK")

In [ ]:
print("Loading master table...")
master = pl.read_parquet(MASTER_FILE)

# Memory: not naive by either definition
memory = master.filter(~pl.col('naive_bio') & ~pl.col('naive_comp'))
# Naive: both definitions agree (strictest set, cleanest negative control for tolerance)
naive  = master.filter(pl.col('naive_bio') & pl.col('naive_comp'))

print(f"Master:  {master.shape}")
print(f"Memory:  {memory.height:,}")
print(f"Naive:   {naive.height:,}")

# Isotype class collapse (subclasses are sequencing artifacts)
def add_isotype_class(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        pl.when(pl.col('c_gene:0').str.starts_with('IGHG')).then(pl.lit('IgG'))
        .when(pl.col('c_gene:0').str.starts_with('IGHA')).then(pl.lit('IgA'))
        .when(pl.col('c_gene:0') == 'IGHM').then(pl.lit('IgM'))
        .when(pl.col('c_gene:0') == 'IGHE').then(pl.lit('IgE'))
        .when(pl.col('c_gene:0') == 'IGHD').then(pl.lit('IgD'))
        .otherwise(pl.lit('other'))
        .alias('isotype_class')
    )

memory = add_isotype_class(memory)
naive  = add_isotype_class(naive)

print(f"\nMemory isotype class distribution:")
print(memory.group_by('isotype_class').agg(pl.len().alias('n')).sort('n', descending=True))

# Confirm key columns
required = ['junction_aa:0', 'cdrh3_length', 'v_gene:0', 'donor', 'lineage']
missing = [c for c in required if c not in master.columns]
if missing:
    print(f"MISSING: {missing}")
else:
    print("\nRequired columns present ✓")

## R1 — CDRH3 Feature-Based Reactivity Model

**Setup:** Logistic regression predicting P(memory) from CDRH3 biophysical features + V-gene indicators. Features associated with autoreactivity will have NEGATIVE coefficients (fewer such sequences survive to memory). Φ_R is defined as the log-odds of being naive-like:

```
logit(P_mem) = α·H_H3 + β·Q_H3 + γ·L_H3 + δ·Y_H3 + Σ_k ξ_k·v_gene_k + intercept
Φ_R(x)       = −logit(P_mem(x))   [high = naive-like features = high reactivity risk]
```

**IGHV4-34 control:** Added as a binary indicator. IGHV4-34 encodes anti-I antigen and anti-H antigen autoreactivity via CDR1/CDR2 (framework-mediated, not CDRH3-dependent). The ξ_{IGHV4-34} coefficient captures the germline-level autoreactivity contribution independently of CDRH3 features. A negative ξ_{IGHV4-34} (fewer IGHV4-34 sequences in memory than expected from CDRH3 features alone) validates the model.

**Sampling strategy:** Balanced subsample of 200k naive + 200k memory sequences (stratified by V-gene to avoid germline frequency confounds), then apply fitted model to all sequences.

In [ ]:
# ── Kyte-Doolittle hydrophobicity scale ──────────────────────────────────────
KD_SCALE = {
    'A':  1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C':  2.5,
    'Q': -3.5, 'E': -3.5, 'G': -0.4, 'H': -3.2, 'I':  4.5,
    'L':  3.8, 'K': -3.9, 'M':  1.9, 'F':  2.8, 'P': -1.6,
    'S': -0.8, 'T': -0.7, 'W': -0.9, 'Y': -1.3, 'V':  4.2,
}
AROMATIC    = set('FWY')
POS_CHARGED = set('KRH')
NEG_CHARGED = set('DE')

def cdrh3_features_vec(junction_aa_series: pl.Series) -> pl.DataFrame:
    """
    Vectorised CDRH3 feature computation from junction_aa strings.
    junction_aa includes flanking Cys (pos 104) and Trp/Phe (pos 118);
    CDR3 proper = junction_aa[1:-1].
    Returns DataFrame with H_H3, Q_H3, L_H3, Y_H3 columns.
    """
    rows = []
    for jaa in junction_aa_series.to_list():
        if jaa is None or len(jaa) <= 2:
            rows.append({'H_H3': None, 'Q_H3': None, 'L_H3': None, 'Y_H3': None})
            continue
        cdr3 = jaa[1:-1]  # strip anchor Cys and Trp/Phe
        n = len(cdr3)
        if n == 0:
            rows.append({'H_H3': None, 'Q_H3': None, 'L_H3': None, 'Y_H3': None})
            continue
        H = sum(KD_SCALE.get(aa, 0.0) for aa in cdr3) / n
        Q = sum(
            1 if aa in POS_CHARGED else (-1 if aa in NEG_CHARGED else 0)
            for aa in cdr3
        )
        Y = sum(1 for aa in cdr3 if aa in AROMATIC) / n
        rows.append({'H_H3': float(H), 'Q_H3': float(Q), 'L_H3': float(n), 'Y_H3': float(Y)})
    return pl.DataFrame(rows, schema={'H_H3': pl.Float64, 'Q_H3': pl.Float64,
                                       'L_H3': pl.Float64, 'Y_H3': pl.Float64})


# Compute features for memory and naive
print("Computing CDRH3 features for memory...")
mem_cols = memory.select([KEY, 'v_gene:0', 'isotype_class', 'junction_aa:0', 'donor', 'lineage'])
mem_feat = cdrh3_features_vec(mem_cols['junction_aa:0'])
mem_feat_df = pl.concat([mem_cols, mem_feat], how='horizontal').filter(
    pl.col('H_H3').is_not_null()
)
print(f"  Memory with valid CDRH3 features: {mem_feat_df.height:,}")

print("Computing CDRH3 features for naive...")
nai_cols = naive.select([KEY, 'v_gene:0', 'isotype_class', 'junction_aa:0', 'donor'])
nai_feat = cdrh3_features_vec(nai_cols['junction_aa:0'])
nai_feat_df = pl.concat([nai_cols, nai_feat], how='horizontal').filter(
    pl.col('H_H3').is_not_null()
)
print(f"  Naive with valid CDRH3 features: {nai_feat_df.height:,}")

# Summary statistics
print("\nCDRH3 feature comparison (memory vs naive):")
for feat in ['H_H3', 'Q_H3', 'L_H3', 'Y_H3']:
    m = mem_feat_df[feat].mean()
    n = nai_feat_df[feat].mean()
    print(f"  {feat}:  memory={m:.3f}  naive={n:.3f}  Δ={m-n:+.3f}")

In [ ]:
# ── Balanced subsample for logistic regression training ──────────────────────
TRAIN_N = 200_000   # sequences per class for training
np.random.seed(42)

# Get top V-genes (≥500 sequences in each of naive and memory for one-hot encoding)
top_vgenes_mem = set(
    mem_feat_df.group_by('v_gene:0').agg(pl.len().alias('n'))
    .filter(pl.col('n') >= 500)['v_gene:0'].to_list()
)
top_vgenes_nai = set(
    nai_feat_df.group_by('v_gene:0').agg(pl.len().alias('n'))
    .filter(pl.col('n') >= 500)['v_gene:0'].to_list()
)
top_vgenes = sorted(top_vgenes_mem & top_vgenes_nai)
print(f"V-genes in model (≥500 in both compartments): {len(top_vgenes)}")

def make_feature_matrix(df: pl.DataFrame, top_vgenes: list, label: int) -> tuple:
    """Build (X, y) from a feature dataframe."""
    rows = df.to_dicts()
    X_list = []
    for r in rows:
        vg = r.get('v_gene:0', '')
        vg_vec = [1.0 if vg == g else 0.0 for g in top_vgenes]
        ighv434 = 1.0 if vg == 'IGHV4-34' else 0.0
        X_list.append([
            r['H_H3'], r['Q_H3'], r['L_H3'], r['Y_H3'], ighv434
        ] + vg_vec)
    X = np.array(X_list, dtype=np.float32)
    y = np.full(len(X_list), label, dtype=np.int8)
    return X, y

# Subsample
mem_train = mem_feat_df.sample(n=min(TRAIN_N, mem_feat_df.height), seed=42)
nai_train = nai_feat_df.sample(n=min(TRAIN_N, nai_feat_df.height), seed=42)

print(f"Training set: {mem_train.height:,} memory + {nai_train.height:,} naive")

X_mem, y_mem = make_feature_matrix(mem_train, top_vgenes, label=1)
X_nai, y_nai = make_feature_matrix(nai_train, top_vgenes, label=0)

X_train = np.vstack([X_nai, X_mem])
y_train = np.concatenate([y_nai, y_mem])

print("Fitting logistic regression (L2, liblinear)...")
# Only standardise the 4 continuous features (cols 0-3); binary indicators unscaled
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_train_scaled[:, :4] = scaler.fit_transform(X_train[:, :4])

clf = LogisticRegression(C=1.0, solver='liblinear', max_iter=500, random_state=42)
clf.fit(X_train_scaled, y_train)

# Training AUC
auc = roc_auc_score(y_train, clf.predict_proba(X_train_scaled)[:, 1])
print(f"Training AUC: {auc:.4f}")

# Feature names and coefficients
feat_names = ['H_H3', 'Q_H3', 'L_H3', 'Y_H3', 'IGHV4-34'] + top_vgenes
coef_df = pl.DataFrame({
    'feature': feat_names,
    'coef': clf.coef_[0].tolist(),
})
print("\nCDRH3 feature coefficients (positive = more memory = lower reactivity risk):")
print(coef_df.head(6))  # first 5 are the key features + IGHV4-34
print(f"  IGHV4-34 coefficient: {coef_df.filter(pl.col('feature')=='IGHV4-34')['coef'][0]:.4f}")

In [ ]:
# ── Apply model to all memory and naive sequences ─────────────────────────────
# Phi_R = -logit(P_memory) = log(P_naive / P_memory)
# High Phi_R: naive-like features → high autoreactivity risk
# Low Phi_R: memory-like features → low autoreactivity risk

def compute_phi_r(feat_df: pl.DataFrame, scaler, clf, top_vgenes: list) -> np.ndarray:
    """Compute Phi_R for all rows of feat_df."""
    rows = feat_df.to_dicts()
    X_list = []
    for r in rows:
        vg = r.get('v_gene:0', '')
        vg_vec = [1.0 if vg == g else 0.0 for g in top_vgenes]
        ighv434 = 1.0 if vg == 'IGHV4-34' else 0.0
        X_list.append([r['H_H3'], r['Q_H3'], r['L_H3'], r['Y_H3'], ighv434] + vg_vec)
    X = np.array(X_list, dtype=np.float32)
    X[:, :4] = scaler.transform(X[:, :4])
    log_odds = clf.decision_function(X)  # logit(P_memory)
    phi_r = -log_odds  # negate: high = naive-like = risky
    return phi_r


print("Computing Phi_R for all memory sequences...")
phi_r_mem = compute_phi_r(mem_feat_df, scaler, clf, top_vgenes)

mem_phi_r_df = mem_feat_df.with_columns(
    pl.Series('phi_R', phi_r_mem)
)

print(f"Memory Phi_R statistics:")
print(mem_phi_r_df['phi_R'].describe())

# Compute for naive too (needed for R2 shift analysis)
print("\nComputing Phi_R for all naive sequences...")
phi_r_nai = compute_phi_r(nai_feat_df, scaler, clf, top_vgenes)

nai_phi_r_df = nai_feat_df.with_columns(
    pl.Series('phi_R', phi_r_nai)
)

print(f"Naive Phi_R statistics:")
print(nai_phi_r_df['phi_R'].describe())

# Save memory Phi_R
mem_phi_r_df.write_parquet(TABLES / "phi_r_scores.parquet")
print(f"\nSaved → {TABLES}/phi_r_scores.parquet")

In [ ]:
# ── Plot R1: feature distributions and model coefficients ─────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Top row: feature distributions naive vs memory
feature_labels = {
    'H_H3': 'CDR3 mean hydrophobicity (KD)',
    'Q_H3': 'CDR3 net charge (K+R+H − D−E)',
    'L_H3': 'CDR3 length (aa)',
    'Y_H3': 'CDR3 aromatic fraction (F+W+Y)',
}

for idx, (feat, label) in enumerate(feature_labels.items()):
    ax = axes[idx // 2][idx % 3] if idx < 4 else None
    if ax is None:
        break
    m_arr = mem_feat_df[feat].to_numpy()
    n_arr = nai_feat_df[feat].to_numpy()
    lo = min(np.percentile(m_arr, 1), np.percentile(n_arr, 1))
    hi = max(np.percentile(m_arr, 99), np.percentile(n_arr, 99))
    bins = np.linspace(lo, hi, 50)
    ax.hist(np.clip(n_arr, lo, hi), bins=bins, density=True,
            color='#90CAF9', alpha=0.6, label='Naive')
    ax.hist(np.clip(m_arr, lo, hi), bins=bins, density=True,
            color='#EF5350', alpha=0.6, label='Memory')
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.set_title(f'{feat} distribution\n(naive vs memory)')
    ax.legend(fontsize=8)

# Bottom middle: coefficient bar chart for 4 CDRH3 features + IGHV4-34
ax5 = axes[1][1]
key_feats = ['H_H3', 'Q_H3', 'L_H3', 'Y_H3', 'IGHV4-34']
key_coefs = [coef_df.filter(pl.col('feature') == f)['coef'][0] for f in key_feats]
colors = ['#1E88E5' if c > 0 else '#E53935' for c in key_coefs]
ax5.barh(key_feats, key_coefs, color=colors, alpha=0.8)
ax5.axvline(0, color='gray', linestyle='--', lw=1)
ax5.set_xlabel('Logistic regression coefficient\n(positive = memory-enriched = lower risk)')
ax5.set_title('Reactivity feature weights\n(logit P_memory; IGHV4-34 = germline control)')

# Bottom right: Phi_R distribution naive vs memory
ax6 = axes[1][2]
bins_r = np.linspace(-6, 6, 80)
ax6.hist(np.clip(phi_r_nai, -6, 6), bins=bins_r, density=True,
         color='#90CAF9', alpha=0.6, label='Naive')
ax6.hist(np.clip(phi_r_mem, -6, 6), bins=bins_r, density=True,
         color='#EF5350', alpha=0.6, label='Memory')
ax6.axvline(0, color='gray', linestyle='--', lw=1)
ax6.set_xlabel('Φ_R  [−logit(P_memory)]')
ax6.set_ylabel('Density')
ax6.set_title('Φ_R: naive vs memory\n(high = naive-like = reactive risk)')
ax6.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES / "fig_r1_phi_r_model.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

# Also save coefficient table
coef_df.write_csv(TABLES / "phi_r_model_coefficients.csv")
print(f"Coefficients saved → {TABLES}/phi_r_model_coefficients.csv")

## R2 — Per-Germline Φ_R Shift (Naive → Memory)

For each IGHV germline gene, compare the mean Φ_R in the naive compartment to the mean Φ_R in the memory compartment. A large **negative** shift (memory Φ_R < naive Φ_R) means: sequences from this germline with autoreactive CDRH3 features were preferentially removed from the repertoire during maturation.

**Expected pattern:**
- **IGHV4-34**: largest negative shift — known autoreactivity via CDR1/CDR2 framework (anti-I/H antigen). The germline indicator captures this, and IGHV4-34 sequences should be strongly depleted in memory.
- **Germlines producing bnAb precursors (IGHV1-2, IGHV1-69)**: may show POSITIVE shift (these germlines produce broadly protective antibodies that are enriched in GC output, which can push memory Φ_R lower than expected under pure depletion).
- **Most germlines**: small or near-zero shift (random selection, not autoreactivity-driven).

**Unit of analysis:** All memory sequences and all naive sequences with valid CDRH3 features. Germlines with < 500 sequences in either compartment are excluded.

**Isotype stratification:** Also computed separately for IgG memory vs IgM memory to assess whether tolerance-depleted autoreactive signatures are more prominent in class-switched (deep-GC) cells.

In [ ]:
# ── Per-germline mean Phi_R in naive and memory ───────────────────────────────
MIN_SEQS = 500

# Naive germline stats
nai_germ = (
    nai_phi_r_df
    .group_by('v_gene:0')
    .agg([
        pl.col('phi_R').mean().alias('mean_phi_R_naive'),
        pl.col('phi_R').std().alias('std_phi_R_naive'),
        pl.len().alias('n_naive'),
    ])
)

# Memory germline stats (all memory)
mem_germ = (
    mem_phi_r_df
    .group_by('v_gene:0')
    .agg([
        pl.col('phi_R').mean().alias('mean_phi_R_memory'),
        pl.col('phi_R').std().alias('std_phi_R_memory'),
        pl.len().alias('n_memory'),
    ])
)

# Memory stratified by isotype class (IgG only)
mem_igg_germ = (
    mem_phi_r_df.filter(pl.col('isotype_class') == 'IgG')
    .group_by('v_gene:0')
    .agg([
        pl.col('phi_R').mean().alias('mean_phi_R_IgG'),
        pl.len().alias('n_IgG'),
    ])
)

# Join and compute shift
germline_shift = (
    nai_germ
    .join(mem_germ, on='v_gene:0', how='inner')
    .join(mem_igg_germ, on='v_gene:0', how='left')
    .filter(
        (pl.col('n_naive') >= MIN_SEQS) & (pl.col('n_memory') >= MIN_SEQS)
    )
    .with_columns([
        (pl.col('mean_phi_R_memory') - pl.col('mean_phi_R_naive'))
        .alias('delta_phi_R'),
        (pl.col('mean_phi_R_IgG') - pl.col('mean_phi_R_naive'))
        .alias('delta_phi_R_IgG'),
        (pl.col('v_gene:0') == 'IGHV4-34').alias('is_ighv434'),
    ])
    .sort('delta_phi_R')
)

germline_shift.write_csv(TABLES / "phi_r_shift_by_germline.csv")
print(f"Germlines with ≥{MIN_SEQS} in both compartments: {germline_shift.height}")
print(f"\nTop 10 most depleted germlines (largest negative Phi_R shift):")
print(germline_shift.head(10).select(['v_gene:0', 'delta_phi_R', 'delta_phi_R_IgG',
                                       'n_naive', 'n_memory', 'is_ighv434']))
print(f"\nIGHV4-34 shift:")
print(germline_shift.filter(pl.col('v_gene:0') == 'IGHV4-34'))

In [ ]:
# ── Plot R2: ranked bar plot with IGHV4-34 highlighted ────────────────────────
top_n = min(40, germline_shift.height)
genes  = germline_shift['v_gene:0'].to_list()[:top_n]
deltas = germline_shift['delta_phi_R'].to_numpy()[:top_n]
is_434 = germline_shift['is_ighv434'].to_numpy()[:top_n]

# Colors: orange = IGHV4-34 (autoreactivity control), blue = depletion, red = enrichment
colors = []
for d, flag in zip(deltas, is_434):
    if flag:
        colors.append('#FF6F00')  # orange = IGHV4-34
    elif d < 0:
        colors.append('#1E88E5')  # blue = depleted in memory (autoreactivity signal)
    else:
        colors.append('#E53935')  # red = enriched in memory

fig, axes = plt.subplots(1, 2, figsize=(18, max(8, top_n * 0.3)))

# Left: delta_phi_R (all memory)
ax = axes[0]
ax.barh(range(top_n), deltas[::-1], color=colors[::-1], alpha=0.85)
ax.set_yticks(range(top_n))
ax.set_yticklabels(genes[::-1], fontsize=8)
ax.axvline(0, color='gray', linestyle='--', lw=1)
ax.set_xlabel('ΔΦ_R (memory − naive)')
ax.set_title('Per-germline reactivity risk shift\n(negative = autoreactive sequences depleted in memory)')

# Add IGHV4-34 annotation
if 'IGHV4-34' in genes:
    idx434 = top_n - 1 - genes.index('IGHV4-34')
    delta434 = deltas[genes.index('IGHV4-34')]
    ax.annotate('IGHV4-34\n(autoreactivity\ncontrol)',
                xy=(delta434, idx434), xytext=(delta434 - 0.3, idx434 + 2),
                fontsize=7, color='#FF6F00',
                arrowprops=dict(arrowstyle='->', color='#FF6F00'))

# Legend
legend_handles = [
    mpatches.Patch(color='#1E88E5', alpha=0.85, label='Depleted in memory (Φ_R ↓)'),
    mpatches.Patch(color='#E53935', alpha=0.85, label='Enriched in memory (Φ_R ↑)'),
    mpatches.Patch(color='#FF6F00', alpha=0.85, label='IGHV4-34 (autoreactivity control)'),
]
ax.legend(handles=legend_handles, fontsize=8, loc='lower right')

# Right: scatter naive vs memory mean Phi_R per germline
ax2 = axes[1]
nai_means  = germline_shift['mean_phi_R_naive'].to_numpy()
mem_means  = germline_shift['mean_phi_R_memory'].to_numpy()
is_434_all = germline_shift['is_ighv434'].to_numpy()
sc_colors  = ['#FF6F00' if f else '#1E88E5' for f in is_434_all]

ax2.scatter(nai_means, mem_means, c=sc_colors, s=20, alpha=0.7, zorder=3)
lim = max(abs(nai_means).max(), abs(mem_means).max()) * 1.05
ax2.plot([-lim, lim], [-lim, lim], 'k--', lw=0.8, label='No shift')
ax2.set_xlabel('Mean Φ_R (naive)')
ax2.set_ylabel('Mean Φ_R (memory)')
ax2.set_title('Naive vs memory mean Φ_R per germline\n(below diagonal = depletion in memory)')

# Label IGHV4-34
if is_434_all.any():
    idx = np.where(is_434_all)[0][0]
    ax2.annotate('IGHV4-34', xy=(nai_means[idx], mem_means[idx]),
                 xytext=(nai_means[idx] + 0.1, mem_means[idx] - 0.2),
                 fontsize=8, color='#FF6F00')

plt.tight_layout()
plt.savefig(FIGURES / "fig_r2_phi_r_shift_by_germline.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

# Additional: Phi_R by isotype class for memory
fig2, ax3 = plt.subplots(figsize=(8, 5))
isotypes = ['IgM', 'IgA', 'IgG']
colors_iso = {'IgM': '#90CAF9', 'IgA': '#66BB6A', 'IgG': '#EF5350'}
bins_r2 = np.linspace(-5, 5, 60)

ax3.hist(np.clip(phi_r_nai, -5, 5), bins=bins_r2, density=True,
         color='#B0BEC5', alpha=0.5, label='Naive (all IgM/D)')
for iso in isotypes:
    arr = mem_phi_r_df.filter(pl.col('isotype_class') == iso)['phi_R'].to_numpy()
    if len(arr) > 100:
        ax3.hist(np.clip(arr, -5, 5), bins=bins_r2, density=True,
                 color=colors_iso[iso], alpha=0.5, label=f'Memory {iso} (n={len(arr):,})')
ax3.axvline(0, color='gray', linestyle='--', lw=1)
ax3.set_xlabel('Φ_R  [−logit(P_memory)]')
ax3.set_ylabel('Density')
ax3.set_title('Φ_R by isotype class\n(IgG deepest GC maturation, expected lowest risk)')
ax3.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIGURES / "fig_r2_phi_r_by_isotype.png", dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")

## Step 4 Summary

| Calculation | Output table | Output figure(s) |
|-------------|-------------|------------------|
| R1: CDRH3 reactivity model | `phi_r_scores.parquet`, `phi_r_model_coefficients.csv` | `fig_r1_phi_r_model.png` |
| R2: per-germline shift | `phi_r_shift_by_germline.csv` | `fig_r2_phi_r_shift_by_germline.png`, `fig_r2_phi_r_by_isotype.png` |

**Key formulae:**
```
H_H3 = mean KD hydrophobicity over CDR3 residues (junction_aa[1:-1])
Q_H3 = net charge = Σ(K,R,H) − Σ(D,E)  over CDR3
L_H3 = CDR3 length (aa, excluding anchor Cys and Trp/Phe)
Y_H3 = (# F + W + Y residues) / L_H3

logit(P_mem) = α·H_H3 + β·Q_H3 + γ·L_H3 + δ·Y_H3 + ξ·IGHV4-34 + Σ_k ξ_k·vgene_k
Φ_R(x)       = −logit(P_mem(x))   [high = naive-like = high reactivity risk]
```

**Key validation:**
- IGHV4-34 coefficient should be negative (depleted in memory = autoreactive) ✓/✗
- H_H3 coefficient: sign determines whether hydrophobic CDR3s are memory-depleted (autoreactivity) or enriched (affinity selection bias)
- Q_H3 coefficient: positive charge → anti-DNA risk → expected negative coefficient
- IgG memory should show the lowest mean Φ_R (deepest GC maturation → most thoroughly tolerized)

**Adaptations from Step 3:**
- Isotype subclasses collapsed throughout (IGHG1/2/3/4 → IgG, IGHA1/2 → IgA)
- Memory compartment defined as ~(naive_bio) & ~(naive_comp); IgM memory included (Step 3 confirmed 55.3% of memory is IgM)
- IGHV4-39 / IGHV3-23 (strongest Φ_A shift from Step 3) included as V-gene covariates in the regression

**Next step:** `05_lagrange.ipynb` — Estimate Lagrange multipliers λ_S and λ_R; compute per-sequence ΔΦ contributions; generate the optimized antibody recommendations.